# Stage 1 (Part 2): MiniMind SFT (指令微調) 實戰

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EndeavorYen/LLM-learning/blob/main/notebooks/01_stage1_minimind/02_minimind_sft_colab.ipynb)

本 Notebook 帶領你完成 Stage 1 下半場的核心目標：
1. **觀察真實 SFT 資料集與 Chat Template**（`<|im_start|>user` 與 `<|im_start|>assistant`）
2. **解剖 MiniMind 的 `SFTDataset.generate_labels`**：親眼看到每個 Token 如何被打上 `-100` 或保留計分
3. **基於 Pre-train 權重 (`pretrain_512.pth`) 執行 `train_full_sft.py`**
4. **對比 Pre-train 與 SFT 的回答差異**（從「胡亂接話」變成「聽懂問題並回答的 AI 助手」）

## 1. 準備 MiniMind 環境與串流下載 SFT 精選資料集 (5,000 筆)
為了節省下載時間與空間，我們使用 HTTP Streaming 直接從 Hugging Face 串流擷取 `sft_t2t_mini.jsonl` 的前 **5,000 筆** 多輪問答劇本（僅需約 2 秒）。

In [ ]:
import os, requests

if not os.path.exists('/content/minimind'):
    !git clone --filter=blob:none https://github.com/jingyaogong/minimind.git /content/minimind && git -C /content/minimind checkout -q 233d00fa068ec2b8c0d4f07b703934d6391a5a73  # [audit #17] pinned
%cd /content/minimind

os.makedirs('/content/minimind/dataset', exist_ok=True)
sft_fast_path = '/content/minimind/dataset/sft_fast_5k.jsonl'

if not os.path.exists(sft_fast_path):
    url = 'https://huggingface.co/datasets/jingyaogong/minimind_dataset/resolve/main/sft_t2t_mini.jsonl'
    with requests.get(url, stream=True) as r, open(sft_fast_path, 'wb') as f:
        r.raise_for_status()
        for i, line in enumerate(r.iter_lines()):
            if i >= 5000:
                break
            if line:
                f.write(line + b'\n')
print(f'✅ SFT 5,000 筆資料集已就緒：{sft_fast_path}')

## 2. 拆解 SFTDataset：親眼看看 `-100` 遮罩如何打在真實對話上

In [ ]:
import sys
sys.path.insert(0, '/content/minimind')
from transformers import AutoTokenizer
from dataset.lm_dataset import SFTDataset

tokenizer = AutoTokenizer.from_pretrained('/content/minimind/model')
ds = SFTDataset(sft_fast_path, tokenizer, max_length=128)
input_ids, labels = ds[0]

print('=== 單筆樣本的 Token 與 Label 對照 (前 40 個 Token) ===')
for idx in range(40):
    tok_id = input_ids[idx].item()
    lbl_id = labels[idx].item()
    tok_str = repr(tokenizer.decode([tok_id]))
    status = '❌ 忽略 (-100)' if lbl_id == -100 else f'✅ 計分 (目標 ID={lbl_id})'
    print(f'Pos {idx:2d}: Token={tok_str:16s} | {status}')

## 3. 執行 Full SFT 微調 (`train_full_sft.py`)

In [ ]:
%cd /content/minimind/trainer
!python train_full_sft.py \
    --data_path ../dataset/sft_fast_5k.jsonl \
    --from_weight pretrain \
    --hidden_size 512 \
    --num_hidden_layers 8 \
    --epochs 1 \
    --batch_size 32 \
    --accumulation_steps 2 \
    --learning_rate 5e-4 \
    --log_interval 25 \
    --save_interval 100

## 4. 推論測試：SFT 微調後的問答表現 (`full_sft_512.pth`)

In [ ]:
%cd /content/minimind
!echo 0 | python eval_llm.py --weight full_sft --hidden_size 512 --num_hidden_layers 8 --max_new_tokens 64